# LR plots
Final eval loss vs learning rate for W&B runs picked by tags, one line per combination of `group_by` config values.
Circles with solid lines are finished runs; squares with dashed lines are still running (latest eval loss).
Minimal port of `jupyter_notebooks/plots_wandb.ipynb` from the `ctx_scl` branch.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import wandb

sys.path.insert(0, str(Path.cwd().parent))
from src.core.utils import solve_config_lr

In [ ]:
def flatten(d, prefix):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flatten(v, f"{prefix}{k}/"))
        else:
            out[f"{prefix}{k}"] = v
    return out


def get_runs(tags, negative_tags=(), project="ideas_cv/llm-random-test"):
    """Runs with all `tags` and none of `negative_tags`; columns config/..., summary/... and lr."""
    filters = {
        "$and": [{"tags": t} for t in tags]
        + [{"tags": {"$nin": [t]}} for t in negative_tags]
    }
    rows = [
        {
            "id": run.id,
            "name": run.name,
            "state": run.state,
            "tags": run.tags,
            **flatten(run.config, "config/"),
            **flatten(run.summary._json_dict, "summary/"),
        }
        for run in wandb.Api().runs(project, filters=filters)
    ]
    if not rows:
        raise ValueError(f"no runs with tags {tags} and without {list(negative_tags)}")
    df = pd.DataFrame(rows)
    df["lr"] = df["config/trainer/learning_rate"].map(lambda x: solve_config_lr(x)[0])
    print(f"{len(df)} runs")
    return df


def varying_config(df):
    """Config columns that vary between runs but not per run (paths, job ids): candidates for group_by."""
    n_values = {
        c: df[c].astype(str).nunique() for c in df.columns if c.startswith("config/")
    }
    return [
        c
        for c, n in n_values.items()
        if 1 < n < len(df) and c != "config/trainer/learning_rate"
    ]


def plot_loss_vs_lr(
    df, group_by, loss_col="summary/steps/eval/loss", ax=None, title=None, ylim=None
):
    """One line per combination of `group_by` values; runs at the same lr (e.g. seeds) are averaged."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))
    df = df.dropna(subset=[loss_col]).assign(
        finished=lambda d: d["summary/step"] + 1 >= d["config/trainer/n_steps"]
    )
    for i, (key, group) in enumerate(df.groupby(group_by, dropna=False)):
        label = ", ".join(
            f"{col.split('/')[-1]}={val}" for col, val in zip(group_by, key)
        )
        for finished, marker, linestyle, suffix in [
            (True, "o", "-", ""),
            (False, "s", "--", " (running)"),
        ]:
            part = group[group["finished"] == finished]
            if part.empty:
                continue
            means = part.groupby("lr")[loss_col].mean()
            ax.plot(
                means.index,
                means.values,
                marker=marker,
                linestyle=linestyle,
                color=f"C{i}",
                label=label + suffix,
            )
    ax.set_xscale("log", base=2)
    ax.set_xlabel("learning rate")
    ax.set_ylabel(loss_col.removeprefix("summary/"))
    ax.set_title(title or "final loss vs learning rate")
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.grid(True)
    ax.legend()
    return ax

In [ ]:
tags = ["lr_sweep"]
negative_tags = ["remove"]

df = get_runs(tags, negative_tags)
varying_config(df)

In [ ]:
plot_loss_vs_lr(
    df,
    group_by=["config/common/method", "config/common/selection"],
    title=" + ".join(tags),
)
plt.show()